In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

In [2]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend



In [3]:
#creating simple deep agaent
agent = create_deep_agent(model='openai:gpt-5.4')

#under the hood what agent is doing is
agent2 = create_deep_agent(model='openai:gpt-5.4',
backend=StateBackend()
)

results = agent2.invoke(
    {
        "messages":[{
            "role": "user",
            "content":(
                "Create a file at /notes/todo.txt with exactly same content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you have done it."
            )
        }]
    }
)

#then final agent natural language reply
print("\n----- Agent Reply -------------------")
print(results["messages"][-1].content)


----- Agent Reply -------------------
[{'type': 'text', 'text': 'Done.', 'annotations': [], 'id': 'msg_0e5d6963f4ca12ab006ab7be78d9d887d29d68263a6cae0591', 'phase': 'final_answer'}]


In [4]:
results

{'messages': [HumanMessage(content='Create a file at /notes/todo.txt with exactly same content:\n1. Record video\n2. Edit video\n3. Upload video\nThen tell me you have done it.', additional_kwargs={}, response_metadata={}, id='39f1289d-d22a-4ea0-a566-f20eaa75c8c2'),
  AIMessage(content=[{'arguments': '{"file_path":"/notes/todo.txt","content":"1. Record video\\n2. Edit video\\n3. Upload video"}', 'call_id': 'call_AbcuMscGl2nB4V0jVMPgHcfe', 'name': 'write_file', 'type': 'function_call', 'id': 'fc_0e5d6963f4ca12ab006ab7be76caf087d29f22afdd1e73413d', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0e5d6963f4ca12ab006ab7be758bcc87d2b31cbad83979f950', 'created_at': 1790426741.0, 'metadata': {}, 'model': 'gpt-5.4-2026-03-05', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05'}, id='resp_0e5d6963f4ca12ab006ab7be758bcc87d2b31cbad83979f950', tool_calls=[{'name': 'write_file', 'args':

In [7]:
print("\n ------Backend check ------------------")
files = results.get('files',{})

if files:
    print(f"State backend is working - {len(files)} files in status")
    for path, content in files.items():
        print(f"\n {path}\n{'-'*40}\n{content}")
else:
    print("No files found...")




 ------Backend check ------------------
State backend is working - 1 files in status

 /notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-09-26T12:45:43.432169+00:00', 'modified_at': '2026-09-26T12:45:43.432169+00:00'}


In [8]:
#File system backend
from deepagents.backends import FilesystemBackend
ROOT = "."
agent3 = create_deep_agent(model='openai:gpt-5.4',
backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True)
)

results = agent3.invoke(
    {
        "messages":[{
            "role": "user",
            "content":(
                "Create a file at /notes/todo.txt with exactly same content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you have done it."
            )
        }]
    }
)

#then final agent natural language reply
print("\n----- Agent Reply -------------------")
print(results["messages"][-1].content)



----- Agent Reply -------------------
[{'type': 'text', 'text': 'Done.', 'annotations': [], 'id': 'msg_00124ffb4a055b64006ab7c0e7a56c87d2a3f6a7bba12aa103', 'phase': 'final_answer'}]


In [9]:
from pathlib import Path
print("\n---- Backend Check (real filesystem)------------")
disk_path = Path(ROOT)/ "notes" / "todo.txt"
if disk_path.exists():
    print(f"Filesystem is working fine- file exists on disk...")
    print(f"{disk_path.resolve()}\n{'-'*40}")
    print(disk_path.read_text())
else:
    print(f"Expected path is not found at {disk_path.resolve()}")


---- Backend Check (real filesystem)------------
Filesystem is working fine- file exists on disk...
/Users/user/damco/langchainKrish/deep-agents/notes/todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video


In [10]:
fresh_agent = create_deep_agent(model='openai:gpt-5.4',
backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True)
)

followup = fresh_agent.invoke(
    {
        "messages":[{
            "role": "user",
            "content":"Read /notes/todo.txt back to me verbatim."
        }]
        #note no 'files' state is passed in the file is read from disk
    }
)

print(f"\n-------Read back the file with a fresh agent...")
print(followup["messages"][-1].content)


-------Read back the file with a fresh agent...
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'annotations': [], 'id': 'msg_0c78fee48077a192006ab7c324ed9487d2adbc9ae711d95606', 'phase': 'final_answer'}]


In [12]:
#langgraph store
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend

store = InMemoryStore()

store_agent = create_deep_agent(model='openai:gpt-5.4',
backend=StoreBackend(
    #local dev: static namespace. no deployment runtime needed needed.
    # in langsmith deployment you would use the user-identity version
    namespace=lambda rt: ("demo-user",),

),
store=store,
)
print("Agent created with storebackend + static namespaces")


Agent created with storebackend + static namespaces


In [13]:
from uuid import uuid4
thread_1 = {"configurable": {"thread_id":str(uuid4())}}

results = store_agent.invoke(
    {
        "messages":[{
            "role": "user",
            "content":(
                "Create a file at /notes/todo.txt with exactly same content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you have done it."
            )
        }]
    },
    config=thread_1,
)

print("\n----------Agent reply (thread1)-----------------------")
print(results["messages"][-1].content)


----------Agent reply (thread1)-----------------------
[{'type': 'text', 'text': 'Done.', 'annotations': [], 'id': 'msg_0235937e67938977006ab7c601f8c087d29e9c52a4da1db132', 'phase': 'final_answer'}]


In [15]:
thread_2 = {"configurable": {"thread_id":str(uuid4())}}

followup = store_agent.invoke(
    {
        "messages":[{
            "role": "user",
            "content":"Read /notes/todo.txt back to me verbatim."
        }]
    },
    config=thread_2,
)
print("\n----------Agent reply (thread2)-----------------------")
print(followup["messages"][-1].content)



----------Agent reply (thread2)-----------------------
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'annotations': [], 'id': 'msg_045829eba4b3d9e1006ab7c6b540f087d2819272d9915e1a44', 'phase': 'final_answer'}]
